# 🧠 CNN Image Classification: Intel Scene Recognition

A portfolio-quality end-to-end image classification project using **TensorFlow/Keras**.

**Goal:** classify natural scene images into 6 categories from the Intel Image Classification dataset: buildings, forest, glacier, mountain, sea, and street.

## Table of Contents
1. Problem and Dataset
2. Environment Setup
3. Download and Inspect Data
4. Preprocessing Pipeline
5. Baseline CNN From Scratch
6. Training and Learning Curves
7. Test Evaluation
8. Confusion Matrix and Classification Report
9. Prediction Visualization
10. Misclassification Analysis
11. Key Learnings, Interview Questions, Common Mistakes, Further Reading


## 1. Problem and Dataset

Image classification assigns one label to an entire image. In this project, the model receives a natural scene image and predicts the most likely scene class.

The Intel Image Classification dataset is useful for portfolio work because it contains real-world visual variability: lighting, texture, scale, and class overlap. For example, glaciers and mountains often appear together, which makes the problem more realistic than a toy dataset.

```mermaid
flowchart LR
    A[Raw image] --> B[Resize and normalize]
    B --> C[CNN feature extraction]
    C --> D[Dense classifier]
    D --> E[Scene class]
```


In [ ]:
# Reproducibility and common imports
import os
import random
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

plt.rcParams.update({
    "figure.figsize": (10, 6),
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
})


In [ ]:
# Install dependencies if needed.
# KaggleHub can download public Kaggle datasets without manually placing files in the notebook folder.
try:
    import tensorflow as tf
    import kagglehub
    import seaborn as sns
    from sklearn.metrics import classification_report, confusion_matrix
except ImportError:
    %pip install -q tensorflow kagglehub seaborn scikit-learn
    import tensorflow as tf
    import kagglehub
    import seaborn as sns
    from sklearn.metrics import classification_report, confusion_matrix

tf.random.set_seed(SEED)
print("TensorFlow:", tf.__version__)


## 2. Download and Inspect Data

The dataset has a folder-per-class structure, which works naturally with `image_dataset_from_directory`.


In [ ]:
DATASET_PATH = Path(kagglehub.dataset_download("puneet6060/intel-image-classification"))
ROOT = DATASET_PATH / "seg_train" / "seg_train"
TEST_ROOT = DATASET_PATH / "seg_test" / "seg_test"

print("Dataset path:", DATASET_PATH)
print("Training classes:", sorted([p.name for p in ROOT.iterdir() if p.is_dir()]))
print("Test classes:", sorted([p.name for p in TEST_ROOT.iterdir() if p.is_dir()]))


In [ ]:
IMG_SIZE = (150, 150)
BATCH_SIZE = 32

train_ds = tf.keras.utils.image_dataset_from_directory(
    ROOT,
    validation_split=0.2,
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
)
val_ds = tf.keras.utils.image_dataset_from_directory(
    ROOT,
    validation_split=0.2,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
)
test_ds = tf.keras.utils.image_dataset_from_directory(
    TEST_ROOT,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

class_names = train_ds.class_names
class_names


## 3. Visualize Sample Images

A quick visual audit helps detect label quality, class imbalance, and resolution issues before modeling.


In [ ]:
def show_samples(dataset, class_names, rows=3, cols=4):
    images, labels = next(iter(dataset))
    plt.figure(figsize=(14, 9))
    for i in range(rows * cols):
        ax = plt.subplot(rows, cols, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(class_names[int(labels[i])])
        plt.axis("off")
    plt.tight_layout()

show_samples(train_ds, class_names)


## 4. Preprocessing Pipeline

The model uses normalized inputs in `[0, 1]`. Caching and prefetching reduce input bottlenecks during training.


In [ ]:
AUTOTUNE = tf.data.AUTOTUNE
normalization = tf.keras.layers.Rescaling(1./255)

def prepare(ds, shuffle=False):
    if shuffle:
        ds = ds.shuffle(1000, seed=SEED)
    return ds.map(lambda x, y: (normalization(x), y), num_parallel_calls=AUTOTUNE).cache().prefetch(AUTOTUNE)

train_ds_prepared = prepare(train_ds, shuffle=True)
val_ds_prepared = prepare(val_ds)
test_ds_prepared = prepare(test_ds)


## 5. Build a CNN From Scratch

This architecture stacks convolution blocks. Each block learns increasingly abstract features: edges, textures, object parts, then scene-level patterns.


In [ ]:
def build_cnn(input_shape=(150, 150, 3), num_classes=6):
    inputs = tf.keras.Input(shape=input_shape)
    x = tf.keras.layers.Conv2D(32, 3, padding="same", activation="relu")(inputs)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPooling2D()(x)

    x = tf.keras.layers.Conv2D(64, 3, padding="same", activation="relu")(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPooling2D()(x)

    x = tf.keras.layers.Conv2D(128, 3, padding="same", activation="relu")(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPooling2D()(x)

    x = tf.keras.layers.Flatten()(x)
    x = tf.keras.layers.Dropout(0.4)(x)
    x = tf.keras.layers.Dense(128, activation="relu")(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    outputs = tf.keras.layers.Dense(num_classes, activation="softmax")(x)
    return tf.keras.Model(inputs, outputs, name="intel_scene_cnn")

model = build_cnn(num_classes=len(class_names))
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.summary()


## 6. Train the Model

Early stopping prevents unnecessary epochs and restores the best validation checkpoint.


In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)
]

history = model.fit(
    train_ds_prepared,
    validation_data=val_ds_prepared,
    epochs=15,
    callbacks=callbacks,
)


In [ ]:
def plot_history(history):
    metrics = [("accuracy", "Accuracy"), ("loss", "Loss")]
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for ax, (metric, title) in zip(axes, metrics):
        ax.plot(history.history[metric], label=f"train_{metric}", linewidth=2)
        ax.plot(history.history[f"val_{metric}"], label=f"val_{metric}", linewidth=2)
        ax.set_title(title)
        ax.set_xlabel("Epoch")
        ax.legend()
    plt.tight_layout()

plot_history(history)


## 7. Evaluate on Test Data


In [ ]:
test_loss, test_acc = model.evaluate(test_ds_prepared)
print(f"Test accuracy: {test_acc:.3f}")
print(f"Test loss: {test_loss:.3f}")


## 8. Confusion Matrix and Classification Report

The confusion matrix shows which classes are visually overlapping. The classification report adds precision, recall, and F1-score.


In [ ]:
y_true = np.concatenate([y.numpy() for _, y in test_ds])
y_prob = model.predict(test_ds_prepared)
y_pred = y_prob.argmax(axis=1)

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 7))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()

print(classification_report(y_true, y_pred, target_names=class_names))


## 9. Visualize Predictions


In [ ]:
def show_predictions(dataset, model, class_names, n=12):
    images, labels = next(iter(dataset))
    probs = model.predict(images)
    preds = probs.argmax(axis=1)
    plt.figure(figsize=(14, 9))
    for i in range(n):
        ax = plt.subplot(3, 4, i + 1)
        plt.imshow((images[i].numpy() * 255).astype("uint8"))
        color = "green" if preds[i] == labels[i] else "red"
        plt.title(f"Pred: {class_names[preds[i]]}\nTrue: {class_names[int(labels[i])]}", color=color)
        plt.axis("off")
    plt.tight_layout()

show_predictions(test_ds_prepared, model, class_names)


## 10. Common Misclassifications

Common confusions usually happen when classes share visual context:

- **Mountain vs glacier:** both can contain snow, rock, and large horizons.
- **Sea vs coast-like mountain scenes:** water and sky dominate the image.
- **Buildings vs street:** streets often contain building facades.

Better data augmentation, transfer learning, and larger image sizes can reduce these errors.

## Key Learnings

- CNNs learn hierarchical visual features from raw pixels.
- Batch normalization stabilizes training.
- Dropout helps regularize dense layers.
- Confusion matrices reveal class-specific weaknesses.

## Interview Questions

1. Why do CNNs use shared convolution kernels?
2. What is the difference between validation and test data?
3. Why can accuracy be misleading on imbalanced datasets?
4. How does dropout reduce overfitting?

## Common Mistakes

- Normalizing validation but not test data.
- Reading confusion matrices with swapped axes.
- Training too long without monitoring validation loss.
- Using test data during model selection.

## Further Reading

- Keras image classification guide: https://keras.io/examples/vision/image_classification_from_scratch/
- CS231n CNN notes: https://cs231n.github.io/convolutional-networks/
- TensorFlow data performance guide: https://www.tensorflow.org/guide/data_performance
